# Toy Transformer Setup (V=5, L=4)

This notebook records the simplified transformer attention setup using the **same conventions as most transformer papers**:

- Token representations are **row vectors**.
- Matrices multiply on the **right**.
- Attention output is computed as **H = P V**.


## Dimensions

- Vocabulary size: **V = 5**
- Context length: **L = 4**
- Embedding dimension: **d = V + L = 9**

Token representations:

$$
x \in \mathbb{R}^{1 \times d}
$$


## Basis vectors

Standard basis vectors are **row vectors**:

$$
e_i \in \mathbb{R}^{1 \times d}
$$

Example:

$$
e_1 = [1,0,0,0,0,0,0,0,0]
$$


## Token and position representation

Token `t` at position `p` is represented by

$$
x(t,p) = e_t + e_{V+p}
$$

Thus each vector has **two non‑zero coordinates**:

- one for the token
- one for the position


## Stacked input matrix

Stacking token vectors gives

$$
X \in \mathbb{R}^{L \times d}
$$

Each row corresponds to one position in the sequence.


## Linear projections

$$
Q = X W_Q
$$

$$
K = X W_K
$$

$$
V = X W_V
$$

with

$$
W_Q, W_K, W_V \in \mathbb{R}^{d \times d}
$$


## Attention scores

$$
S = Q K^T
$$

Shape:

$$
S \in \mathbb{R}^{L \times L}
$$

Entry $S_{ij}$ is the score between query position $i$ and key position $j$.


## Attention probabilities

Apply row‑wise softmax:

$$
P = \text{softmax}(S)
$$

Each row sums to 1.


## Attention output

$$
H = P V
$$

Shapes:

- $P \in \mathbb{R}^{L \times L}$
- $V \in \mathbb{R}^{L \times d_v}$
- $H \in \mathbb{R}^{L \times d_v}$

Row $i$:

$$
h_i = \sum_{j=1}^{L} P_{ij} v_j
$$


## Final output (optional)

If predicting a token distribution:

$$
z = h_L W_O
$$

$$
p = \text{softmax}(z)
$$

where

$$
W_O \in \mathbb{R}^{d_v \times V}
$$


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Parameters from context
V = 5           # Vocabulary size
L = 4           # Context length
d_model = V + L # Dimension (9)

In [ ]:
def get_input_matrix(token_indices):
    """
    Constructs X in R^{L x d} using torch operations
    """
    # Ensure tokens are a tensor
    tokens = torch.tensor(token_indices)
    
    # Content: e_t
    content_eb = F.one_hot(tokens, num_classes=V).float() # (L, V)
    
    # Position: e_{V+p}
    pos_indices = torch.arange(L)
    pos_eb = F.one_hot(pos_indices, num_classes=L).float() # (L, L)
    
    # Concatenate to form x(t,p) = e_t + e_{V+p}
    X = torch.cat([content_eb, pos_eb], dim=-1) # (L, V+L)
    return X

# Example Sequence: [0, 2, 1, 4]
X = get_input_matrix([0, 2, 1, 4])

In [ ]:
# Standard transformer conventions
# We use bias=False to strictly follow the notation Q = XW_Q
W_Q = torch.nn.Linear(d_model, d_model, bias=False)
W_K = torch.nn.Linear(d_model, d_model, bias=False)
W_V = torch.nn.Linear(d_model, V, bias=False) # d_v = V

# Initialize as identity for the 'Probabilistic Pointer' behavior
with torch.no_grad():
    W_Q.weight.copy_(torch.eye(d_model))
    W_K.weight.copy_(torch.eye(d_model))
    # Identity for the first V components
    W_V.weight.copy_(torch.eye(d_model)[:V, :]) 

Q = W_Q(X)
K = W_K(X)
V_mat = W_V(X)

In [ ]:
# S = Q K^T
scores = torch.matmul(Q, K.transpose(-2, -1))

# Causal Masking
mask = torch.tril(torch.ones(L, L))
scores = scores.masked_fill(mask == 0, float('-inf'))

# P = softmax(S)
P = F.softmax(scores, dim=-1)

# H = P V
H = torch.matmul(P, V_mat)

In [ ]:
# The finite-state machine interpretation
last_token_probs = P[-1, :] # Probability distribution over previous positions

print(f"Attention weights for the last query: {last_token_probs.detach().numpy()}")
print(f"Resulting distribution over vocabulary: {H[-1, :].detach().numpy()}")